# 01. 법인 명단과 기업개황 받기

**이 노트북이 하는 일:** DART에 등록된 모든 법인의 명단을 받고, 그중 **상장사**만 추린 뒤, 각 기업의 **기본 정보(업종·결산월·법인구분)** 를 받아 표로 만듭니다. "어떤 기업의 재무제표를 받을지" 정하는 재료를 만드는 단계입니다.

| 항목 | 내용 |
|---|---|
| 입력 | `data/raw/DART_파일럿/corpCode.zip`, `data/raw/DART_파일럿/company/*.json` (이미 받아 둔 것) |
| 출력 | 화면 요약만. 아무 파일도 덮어쓰지 않음 (기존 `기업개황_상장사전체.csv`와 같은지 비교만 함) |
| DART API 호출 | **기본값은 호출하지 않음**(`ALLOW_API = False`). 받아 둔 파일만 읽음 |
| 실행 시간 | 약 10~20초 (저장된 파일 4,009개를 읽는 시간) |
| 먼저 알아야 할 것 | 00번 노트북 (상태코드, 캐시, 환경변수 키) |
| 원래 스크립트 | `원본_스크립트/01_명단_기업개황_원본.py` (파일럿 폴더에 원본 보관) |

## 이 노트북의 지도
1. 전체 법인 명단(`corpCode.zip`) 열어 보기
2. 상장사만 추리기
3. 기업개황(`company.json`) 한 건 읽어 보기
4. 4,009곳의 기업개황을 표로 모으기 (캐시 규칙)
5. 결과 분포 보기 — 법인구분, 결산월, 업종코드
6. 제조업 추리기
7. 확인하지 못한 것과 문제점

> **안전장치.** 아래 `ALLOW_API`를 `True`로 바꾸지 않는 한, 이 노트북은 DART에 아무 요청도 보내지 않습니다. `True`로 바꾸면 **저장된 파일이 없는 기업만** 새로 요청하고(기업당 1회), 키(`DART_API_KEY`)가 필요합니다.

In [1]:
import io, json, re, sys, time, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import config
from dart_api import CACHEABLE_STATUSES, get_bytes, get_json

ALLOW_API = False          # ← True로 바꾸면 저장 안 된 것만 DART에 요청함. 기본은 False(읽기 전용)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("API 호출 허용:", ALLOW_API)
print("원본 폴더:", "Corp/" + config.RAW.relative_to(config.BASE).as_posix())

API 호출 허용: False
원본 폴더: Corp/data/raw/DART_파일럿


---
# 1. 전체 법인 명단 `corpCode.zip`

DART에 등록된 법인은 상장사만 있는 게 아닙니다. 비상장 회사, 펀드, 해산한 회사까지 **모두** 번호표(`corp_code`)를 갖고 있습니다. 이 번호표 목록을 한 번에 내려받는 것이 `corpCode.xml`이며, ZIP 파일로 옵니다.

저장된 파일이 있으면 그것을 읽고, 없을 때만 (그리고 `ALLOW_API=True`일 때만) 새로 받습니다. 이것이 **캐시** — "한 번 받은 것은 다시 받지 않는다"입니다.

In [2]:
zip_path = config.RAW / "corpCode.zip"

if zip_path.exists():
    zip_bytes = zip_path.read_bytes()
    print("저장된 파일을 읽음:", zip_path.name, f"({len(zip_bytes)/1e6:.1f} MB)")
elif ALLOW_API:
    zip_bytes = get_bytes("corpCode.xml", {})
    zip_path.write_bytes(zip_bytes)
    print("DART에서 새로 받아 저장함")
else:
    raise SystemExit("corpCode.zip이 없고 ALLOW_API=False라서 멈춤")

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    print("ZIP 안의 파일:", zf.namelist())
    xml_name = zf.namelist()[0]
    xml_bytes = zf.read(xml_name)
print(f"XML 크기: {len(xml_bytes)/1e6:.1f} MB")
print()
print("XML 맨 앞 부분(법인 2개):")
print(xml_bytes.decode("utf-8")[:520])

저장된 파일을 읽음: corpCode.zip (3.6 MB)
ZIP 안의 파일: ['CORPCODE.xml']
XML 크기: 30.4 MB

XML 맨 앞 부분(법인 2개):
<?xml version="1.0" encoding="UTF-8"?>
<result>
    <list>
        <corp_code>00434003</corp_code>
        <corp_name>다코</corp_name>
        <corp_eng_name>Daco corporation</corp_eng_name>
        <stock_code> </stock_code>
        <modify_date>20170630</modify_date>
    </list>
    <list>
        <corp_code>00430964</corp_code>
        <corp_name>굿앤엘에스</corp_name>
        <corp_eng_name>Good &amp; LS Co.,Ltd.</corp_eng_name>
        <stock_code> </stock_code>
        <modify_date>20170630</modify_date>
    </list>


### 해석
- ZIP 안에는 `CORPCODE.xml` 하나가 있고, **법인 한 곳이 `<list>…</list>` 한 덩어리**입니다. 칸은 네 개입니다: `corp_code`(DART 번호, 8자리), `corp_name`(이름), `stock_code`(종목코드, 6자리 — **비상장이면 비어 있음**), `modify_date`(이 정보가 마지막으로 바뀐 날).
- XML은 `<이름>값</이름>` 모양으로 적은 글자 형식입니다. 파이썬 `xml.etree`로 읽으면 칸별로 꺼낼 수 있습니다.
- **`stock_code`가 비어 있으면 상장하지 않은 법인**으로 봅니다. 이 한 칸이 다음 단계의 필터입니다. 위 출력에서 보듯 **"비어 있다"는 것이 실제로는 공백 한 칸(`" "`)** 입니다. 그래서 코드가 `.strip()`(앞뒤 공백 제거)을 먼저 하고 "빈 글자인가"를 봅니다. 이걸 빼먹으면 비상장 법인이 전부 상장사로 잡힙니다.
- XML에는 영문 이름(`corp_eng_name`)도 있지만 우리는 쓰지 않습니다.

In [3]:
root = ET.fromstring(xml_bytes)

rows = []
for node in root.findall("list"):
    rows.append({
        "corp_code":   (node.findtext("corp_code") or "").strip(),
        "corp_name":   (node.findtext("corp_name") or "").strip(),
        "stock_code":  (node.findtext("stock_code") or "").strip(),
        "modify_date": (node.findtext("modify_date") or "").strip(),
    })
corp_all = pd.DataFrame(rows)

print(f"전체 법인 수: {len(corp_all):,}")
corp_all.head(5)

전체 법인 수: 119,879


,corp_code,corp_name,stock_code,modify_date
0,00434003,다코,,20170630
1,00430964,굿앤엘에스,,20170630
2,00388953,크레디피아제이십오차유동화전문회사,,20170630
3,00179984,연방건설산업,,20170630
4,00420143,브룩스피알아이오토메이션잉크,,20170630


# 2. 상장사만 추리기

종목코드가 **비어 있지 않은** 법인만 남깁니다.

In [4]:
listed = corp_all[corp_all["stock_code"] != ""].copy()

print(f"종목코드가 있는 법인: {len(listed):,}곳  (전체의 {len(listed)/len(corp_all):.1%})")
print()
print("종목코드가 있는 법인 예시:")
display(listed.sample(5, random_state=config.SEED))
print("종목코드 자릿수 분포:", dict(Counter(listed["stock_code"].str.len())))

종목코드가 있는 법인: 4,009곳  (전체의 3.3%)

종목코드가 있는 법인 예시:


,corp_code,corp_name,stock_code,modify_date
99018,01063954,수젠텍,253840,20240701
115262,01674538,엔에이치기업인수목적24호,437780,20250526
7219,00264556,에이스일렉트로닉스,038690,20170630
108985,00462121,LIG아큐버,073490,20260415
111162,01042979,휴마시스,205470,20260401


종목코드 자릿수 분포: {6: 4009}


### 해석
- 전체 약 12만 개 법인 중 **종목코드가 있는 곳은 4,009곳**입니다.
- 종목코드는 모두 6자리입니다(위 출력 확인).

### ⚠ 이 단계에 있는 알려진 구멍
**"종목코드가 있다 = 지금 상장 중"이 아닙니다.** 4,009곳은 현재 상장사 수(약 2,760곳)보다 많습니다. 즉 **이미 상장폐지된 회사, 스팩, 선박투자회사 등이 섞여 있을 가능성**이 큽니다(이름으로 본 추정이며 확인은 못 했습니다).

반대 방향의 문제도 있습니다. **corpCode.xml이 상장폐지된 법인의 종목코드를 남겨 두는지**는 DART 가이드에 설명이 없습니다. 만약 지워 버린다면, 2019~2021년에 상장해 있다가 이후 폐지된 기업이 **명단에서 처음부터 빠집니다.** 폐지된 기업은 금리 인상으로 크게 나빠진 기업일 수도, 합병으로 사라진 건실한 기업일 수도 있어서, 이들이 빠지면 결과가 **어느 쪽으로든 틀어질 수 있습니다**(방향은 빠진 기업의 노출도와 ICR의 관계에 달려 있고 지금은 알 수 없습니다). 이것이 **생존편향**이고 결정 A9·A14입니다. **A9(표본 처리 규칙)는 2026-11-04에 확정**되었고(사전기간 상장 기업을 모집단으로 하고 이탈 기업은 삭제하지 않음), 명단의 완전성을 거래소(KIND) 목록과 대조하는 **A14는 허락이 필요해서 아직 확인하지 않았습니다.**

---
# 3. 기업개황(`company.json`) 한 건 읽어 보기

명단에는 이름·번호밖에 없습니다. 업종, 결산월, 시장 구분은 **기업개황**에 있고, **기업당 요청 1번**입니다. 삼성전자(corp_code `00126380`)의 저장된 응답을 먼저 열어 봅니다. (회사 대표자명·주소·전화번호 등 우리가 쓰지 않는 칸은 숨겼습니다.)

In [5]:
sample_path = config.RAW / "company" / "00126380.json"
sample = json.loads(sample_path.read_text(encoding="utf-8"))

meaning = {
    "status": "상태코드. 000=정상",
    "message": "상태 메시지",
    "corp_code": "DART 고유번호(8자리)",
    "corp_name": "회사 이름",
    "stock_name": "종목명",
    "stock_code": "종목코드(6자리)",
    "corp_cls": "법인구분. Y=유가증권(코스피), K=코스닥, N=코넥스, E=기타",
    "induty_code": "업종코드. 한국표준산업분류(KSIC) 계열. 2~5자리가 섞여 있음. 앞 2자리가 대분류 단위",
    "est_dt": "설립일(YYYYMMDD)",
    "acc_mt": "결산월. 12 = 12월 결산",
}
pd.DataFrame(
    [(k, sample[k], meaning[k]) for k in meaning if k in sample],
    columns=["칸 이름", "삼성전자의 값", "뜻"],
)

,칸 이름,삼성전자의 값,뜻
0,status,000,상태코드. 000=정상
1,message,정상,상태 메시지
2,corp_code,00126380,DART 고유번호(8자리)
3,corp_name,삼성전자(주),회사 이름
4,stock_name,삼성전자,종목명
5,stock_code,005930,종목코드(6자리)
6,corp_cls,Y,"법인구분. Y=유가증권(코스피), K=코스닥, N=코넥스, E=기타"
7,induty_code,264,업종코드. 한국표준산업분류(KSIC) 계열. 2~5자리가 섞여 있음. 앞 2자리가 ...
8,est_dt,19690113,설립일(YYYYMMDD)
9,acc_mt,12,결산월. 12 = 12월 결산


### 해석
- 우리가 실제로 쓰는 칸은 **`corp_cls`(시장 구분), `induty_code`(업종), `acc_mt`(결산월)** 세 개입니다.
- 삼성전자의 `induty_code`는 **264**입니다. 앞 두 자리 `26`이 업종 대분류 단위이고, 제조업 범위(10~34) 안입니다.
- **업종코드는 자릿수가 섞여 있습니다.** 어느 경우든 앞 2자리로 대분류를 봅니다. 이 규칙(앞 2자리 10~34)은 공식 분류표로 확인되어 A1로 확정되었습니다(근거는 6절 아래 참고).

---
# 4. 4,009곳의 기업개황을 표로 모으기

기업 4,009곳을 한 곳씩 돌면서 **저장된 파일이 있으면 읽고, 없으면** (허용된 경우에만) DART에 요청합니다.

흐름:
```
기업 하나 → 저장된 파일 있나? ─ 있음 → 읽기
                         └ 없음 → (ALLOW_API=True일 때만) DART 요청
                                    → 상태가 000 또는 013이면 파일로 저장
                                    → 0.4초 쉼(서버에 무리 주지 않으려고)
```
저장은 **000·013만** 합니다(00번 노트북에서 본 이유).

In [6]:
def load_company(corp_code):
    # 기업개황 한 건을 돌려준다. (자료, 출처) 출처는 'cache' / 'api' / 'missing'
    cache = config.RAW / "company" / f"{corp_code}.json"
    if cache.exists():
        return json.loads(cache.read_text(encoding="utf-8")), "cache"
    if not ALLOW_API:
        return None, "missing"
    data = get_json("company.json", {"corp_code": corp_code})
    if data.get("status") in CACHEABLE_STATUSES:
        cache.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    time.sleep(config.SLEEP)
    return data, "api"


def name_flags(name):
    # 이름에 스팩/지주회사 모양이 있으면 표시만 한다(삭제하지 않음)
    flags = [k for k, pat in config.FLAG_NAME_PATTERNS.items() if re.search(pat, name)]
    return ",".join(flags)


records, source_count, bad_status = [], Counter(), []
for r in listed.itertuples():
    data, src = load_company(r.corp_code)
    source_count[src] += 1
    if data is None:
        continue
    if data.get("status") != "000":
        bad_status.append((r.corp_code, data.get("status")))
        continue
    records.append({
        "corp_code": r.corp_code,
        "corp_name": r.corp_name,
        "stock_code": r.stock_code,
        "corp_cls": data.get("corp_cls", ""),
        "induty_code": str(data.get("induty_code", "")).strip(),
        "acc_mt": data.get("acc_mt", ""),
        "name_flags": name_flags(r.corp_name),
    })
company = pd.DataFrame(records)

print("읽은 출처:", dict(source_count))
print("상태가 000이 아니어서 표에서 빠진 기업:", len(bad_status))
print("표의 크기:", company.shape)
company.head()

읽은 출처: {'cache': 4009}
상태가 000이 아니어서 표에서 빠진 기업: 0
표의 크기: (4009, 7)


,corp_code,corp_name,stock_code,corp_cls,induty_code,acc_mt,name_flags
0,00260985,한빛네트,036720,E,58221,12,
1,00264529,엔플렉스,040130,E,58221,12,
2,00358545,동서정보기술,055000,E,58221,12,
3,00231567,애드모바일,032600,E,58221,12,
4,00359614,리더컴,056140,E,26410,12,


### 해석
- 출처가 전부 `cache`이므로 **이번 실행에서는 DART에 요청이 한 번도 가지 않았습니다.**
- 이전에 받아 둔 4,009곳이 모두 정상(000)이어서 표에서 빠진 기업은 없습니다.
- 한 가지 확인: 이 표가 **원래 스크립트가 만든 `기업개황_상장사전체.csv`와 같은가?** 같지 않다면 이 노트북 코드가 원본과 다르게 동작한다는 뜻입니다.

In [7]:
original = pd.read_csv(config.OUT / "기업개황_상장사전체.csv", dtype=str, keep_default_na=False)
mine = company.astype(str)

same_shape = original.shape == mine.shape
same_values = original.reset_index(drop=True).equals(mine.reset_index(drop=True))
print("표 크기 같음:", same_shape, original.shape, mine.shape)
print("모든 값 같음:", same_values)

표 크기 같음: True (4009, 7) (4009, 7)
모든 값 같음: True


### 해석
`True / True`이면 이 노트북 코드가 원래 스크립트(`원본_스크립트/01_명단_기업개황_원본.py --all`)와 **같은 결과**를 낸다는 뜻입니다. (값이 다르면 어느 칸이 다른지 따로 찾아봐야 합니다.)

---
# 5. 분포 보기 — 어떤 기업들이 들어 있나

In [8]:
print("[법인구분 corp_cls]  Y 유가증권 / K 코스닥 / N 코넥스 / E 기타")
print(company["corp_cls"].value_counts().to_string())
print()
print("[결산월 acc_mt]")
print(company["acc_mt"].value_counts().to_string())
print()
print("[업종코드 자릿수]")
print(company["induty_code"].str.len().value_counts().sort_index().to_string())
print()
print("[이름으로 표시한 스팩/지주]")
print(company["name_flags"].replace("", "해당 없음").value_counts().to_string())

[법인구분 corp_cls]  Y 유가증권 / K 코스닥 / N 코넥스 / E 기타
corp_cls
K    1825
E    1248
Y     829
N     107

[결산월 acc_mt]
acc_mt
12    3868
03      47
06      40
09      11
02       9
11       8
04       6
10       6
01       5
05       4
08       4
07       1

[업종코드 자릿수]
induty_code
2      44
3    1402
4     483
5    2080

[이름으로 표시한 스팩/지주]
name_flags
해당 없음      3714
spac        188
holding     107


### 해석
- **법인구분 E(기타)가 1,248곳이나 됩니다.** 코스피(Y) 829곳보다 많습니다. E에는 이미 상장폐지된 법인, 스팩, 선박투자회사 등이 섞여 있다고 **추정**되지만 구성은 확인하지 못했습니다. 그래서 사전 고정 문서는 "현재 시장구분(`corp_cls`)으로 기업을 거르지 말고 **사전기간에 보고서를 냈는지**로 정의한다"고 적었습니다. 현재 구분은 과거 상장 여부를 뜻하지 않기 때문입니다.
- **결산월이 12월이 아닌 기업이 있습니다.** 이들은 분기 구분이 달력과 어긋납니다(예: 9월 결산이면 "1분기"가 10~12월). 12월 결산만 쓸지는 결정 A2입니다.
- **업종코드 자릿수는 2자리 44곳, 3자리 1,402곳, 4자리 483곳, 5자리 2,080곳입니다.** 이전 문서에는 "3~5자리"라고 적혀 있었는데, 그것은 150곳만 본 관찰이었고 전체를 보니 **2자리 코드도 있습니다.** 2자리 코드는 앞 2자리가 곧 전체 코드라서 "앞 2자리가 10~34" 규칙은 그대로 적용되지만, 이런 기업이 세분류가 없다는 점은 기억해 둘 만합니다(예: 세부 업종 FE를 쓸 때). 문서의 "3~5자리"는 "2~5자리"로 고쳐야 합니다.
- **12월 결산은 3,868곳, 그 외는 141곳**입니다.
- **스팩/지주 표시는 이름 기준이라 불완전합니다.** 표시만 하고 지우지 않습니다.

---
# 6. 제조업 추리기

규칙: 업종코드 **앞 2자리가 10~34**이면 제조업. (결정 A1, 2026-11-04 확정)

In [9]:
prefixes = config.MANUFACTURING_PREFIXES
company["is_manu"] = company["induty_code"].str[:2].isin(prefixes)
manu = company[company["is_manu"]]

print(f"전체 {len(company):,}곳 중 제조업 {len(manu):,}곳 ({len(manu)/len(company):.1%})")
print()
print("제조업의 법인구분:")
print(manu["corp_cls"].value_counts().to_string())
print()
print(f"제조업 중 12월 결산: {(manu['acc_mt']=='12').sum():,}곳 / 비12월: {(manu['acc_mt']!='12').sum():,}곳")
print(f"제조업 중 이름으로 스팩/지주 표시: {(manu['name_flags']!='').sum()}곳")

전체 4,009곳 중 제조업 2,250곳 (56.1%)

제조업의 법인구분:
corp_cls
K    1125
E     570
Y     491
N      64

제조업 중 12월 결산: 2,208곳 / 비12월: 42곳
제조업 중 이름으로 스팩/지주 표시: 30곳


### 해석
- 제조업은 **2,250곳**이고 이름으로 스팩·지주 표시가 붙은 곳이 30곳 포함되어 있습니다. 이 숫자는 **"분석 표본"이 아니라 "후보 명단"** 입니다. 앞으로 (a) 사전기간 보고서가 있는지 선별하고 (b) 스팩·지주·폐지 기업을 처리하고 (c) 코넥스·비12월 결산을 어떻게 할지 정해야 최종 표본이 됩니다.
- 법인구분 N(코넥스) 64곳은 A2에서 **제외로 확정**되었고, 비12월 결산 42곳도 제외(12월 결산만)로 확정되었습니다(2026-11-04).

In [10]:
# 제조업 2자리 코드별 기업 수 (큰 순서)
by2 = manu["induty_code"].str[:2].value_counts().sort_index()
pd.DataFrame({"기업 수": by2}).T

induty_code,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33
기업 수,99,13,1,30,55,12,6,31,6,6,219,207,69,44,111,72,497,155,135,289,127,36,10,20


### 참고: 앞 2자리가 뜻하는 업종 (공식 분류표로 확인, 2026-11-04)
아래 이름은 **한국표준산업분류 제10차 개정(2017) 공식 표**와 대조해 확인했습니다. 출처: 통계청(현 국가데이터처) 보도자료 「2022년 하반기 지역별고용조사 시군구 주요고용지표」(2023.2.21.) 부록2, 인쇄 쪽 84. 이 표는 분류표 원본이 아니라 공식 보도자료의 부록입니다. 같은 범위(C 제조업 10~34)는 **제11차 개정 해설서**(목차 175쪽, 시행 2024-07-01)와 2022 상반기 보도자료 부록에서도 확인했습니다. DART의 `induty_code`가 어느 개정판을 따르는지에 대한 공식 설명은 찾지 못했습니다. 신·구연계표로 대조하면 제조업 5자리 코드 999곳 중 제11차에만 있는 코드가 41곳, 제10차에만 있는 코드가 0곳이어서 **제11차 코드가 적어도 일부 섞여 있는 것으로 보입니다(정황 근거)**. '109…' 15곳도 옛 코드가 아니라 제11차 신설 소분류입니다. 앞 2자리 10~34 판정에는 영향이 없습니다.

| 코드 | 업종 | 코드 | 업종 |
|---|---|---|---|
| 10 | 식료품 | 23 | 비금속 광물제품 |
| 11 | 음료 | 24 | 1차 금속 |
| 12 | 담배 | 25 | 금속가공제품; 기계 및 가구 제외 |
| 13 | 섬유제품; 의복 제외 | 26 | 전자부품, 컴퓨터, 영상, 음향 및 통신장비 |
| 14 | 의복, 의복 액세서리 및 모피제품 | 27 | 의료·정밀·광학기기·시계 |
| 15 | 가죽, 가방 및 신발 | 28 | 전기장비 |
| 16 | 목재 및 나무제품; 가구 제외 | 29 | 기타 기계 및 장비 |
| 17 | 펄프, 종이 및 종이제품 | 30 | 자동차·트레일러 |
| 18 | 인쇄 및 기록매체 복제업 | 31 | 기타 운송장비 |
| 19 | 코크스·연탄·석유정제품 | 32 | 가구 |
| 20 | 화학물질·화학제품; 의약품 제외 | 33 | 기타 제품 |
| 21 | 의료용 물질·의약품 | **34** | **산업용 기계 및 장비 수리업** |
| 22 | 고무 및 플라스틱제품 | | |

공식표에서 **C 제조업은 중분류 10~34**이고, 34번은 이름이 수리업이지만 제조업 안에 들어 있습니다. 위 출력에서 보듯 **우리 명단에는 앞 2자리가 34인 기업이 한 곳도 없어서**(24개 코드만 나타남: 10~33) 실제 영향은 없습니다. **A1은 "앞 2자리 10~34"로 확정**되었습니다.

---
# 7. 정리

### 이 노트북으로 확인된 것
- 법인 명단 → 상장사 4,009곳 → 제조업 약 2,250곳으로 좁혀지는 과정이 원래 스크립트의 결과와 같은 값으로 재현됩니다(위 비교 결과 참고).
- 이 노트북은 DART에 요청을 보내지 않았습니다.

### 확인하지 못한 것 (문제점)
| 문제 | 이유 | 관련 결정 |
|---|---|---|
| 상장폐지 기업이 명단에 얼마나 있는지, 사전기간 폐지 기업이 빠졌는지 | corpCode.xml의 폐지 법인 처리 설명이 가이드에 없음. KIND 대조는 허락 전 | A9, A14 |
| `induty_code`가 제10차·제11차 중 어느 개정판인지 | 가이드에 설명 없음. 정황상 제11차 코드가 일부 포함. 제조업 범위(10~34)는 두 판 모두 같아 확정 | A1 |
| 법인구분 E의 구성 | 이름 기준 추정뿐 | A9 |
| 비12월 결산·코넥스 처리 | 미정 | A2 |

### 다음 단계
`02_재무제표_수집.ipynb`: 이 후보 명단에서 기업을 정해 재무제표를 받습니다.